# Databricks Mosaic AI Core APIs 

## MLflow – Model & Agent Lifecycle
- **mlflow.pyfunc.log_model**: Packages an agent or model as a deployable MLflow artifact.
- **mlflow.register_model**: Registers a model or agent into Unity Catalog for governance and deployment.
- **mlflow.pyfunc.load_model**: Loads a registered model for batch inference or Python execution.
- **mlflow.set_experiment**: Sets the experiment location for runs, traces, and evaluations.
- **@mlflow.trace**: Automatically captures agent steps, tool calls, and LLM interactions.

## MLflow GenAI – Evaluation
- **mlflow.genai.evaluate**: Runs offline evaluation of agents or RAG pipelines using LLM judges.
- **mlflow.genai.datasets.create_dataset**: Creates a reusable dataset for agent evaluation.
- **mlflow.genai.datasets.get_dataset**: Loads an evaluation dataset for scoring and regression testing.
- **mlflow.genai.scorers.Correctness**: Scores factual correctness of agent responses.
- **mlflow.genai.scorers.RelevanceToQuery**: Scores how relevant a response is to the input query.
- **mlflow.genai.scorers.Guidelines**: Scores compliance with custom safety or policy rules.

## Databricks Agent Framework – Deployment
- **agents.deploy**: Deploys a Unity-Catalog-registered agent to Mosaic AI Model Serving.
- **agents.list_deployments**: Lists all active agent deployments.
- **agents.get_deployment**: Retrieves metadata and status for a deployed agent.
- **agents.delete_deployment**: Removes an agent deployment and serving endpoint.

## Unity Catalog – Governed Tools
- **DatabricksFunctionClient**: Client for registering and invoking Unity Catalog function tools.
- **create_python_function**: Registers a Python function as a governed agent tool.
- **create_function**: Registers a SQL function as a governed agent tool.

## Runtime Execution
- **mlflow.models.predict**: Executes a model or agent via its standardized predict interface.
- **mlflow.pyfunc.spark_udf**: Runs a model or agent at scale inside Spark SQL.

# Agent Evaluation
Now that we've created an agent with tools, we'll evaluate it

In [ ]:
%pip install -U -qqqq "mlflow-skinny[databricks]>=3.14" "databricks-langchain>=0.19.0" databricks-agents langchain-openai twilio python-dotenv
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
sys.path.append("./tools")

# Populate os.environ from job/notebook widgets BEFORE importing weatherwise_agent
# (the agent reads os.environ at import time). Interactive runs without widgets fall
# back to the .env file loaded in the next cell. Widget values take precedence.
_CONFIG_KEYS = [
    "TARGET_CATALOG", "TARGET_SCHEMA", "VS_INDEX", "VS_INDEX_BASE_TABLE",
    "DATABRICKS_HOST", "AI_GATEWAY_MODEL", "AI_GATEWAY_BASE_URL",
    "MLFLOW_EXPERIMENT", "RETRIEVER_TOOL_NAME", "AGENT_NAME",
    "MAILGUN_API_URL", "MAILGUN_API_KEY", "SENDER", "RECIPIENT",
    "SMS_ACCOUNT_SID", "SMS_AUTH_TOKEN", "SMS_ACCOUNT_ID", "LLM_DESTINATION_MODEL", "AGENT_LLM_SECRET_SCOPE", "WAREHOUSE_ID", "GENIE_ID", "AGENT_SCALE_TO_ZERO",
]
for _k in _CONFIG_KEYS:
    try:
        _v = dbutils.widgets.get(_k)
    except Exception:
        _v = ""
    if _v:
        os.environ[_k] = _v


In [0]:
import os
from dotenv import load_dotenv, find_dotenv
import weatherwise_agent
load_dotenv(find_dotenv())  

def redact(val, show=4):
    if not val or len(val) <= show*2:
        return "****"
    return f"{val[:show]}...{val[-show:]}"

# Extract the names of all tools defined in the module
tool_names = [t.name for t in weatherwise_agent.tools]

print("Available tools:")
for name in tool_names:
    print(f" - {name}")

# read OS variables into notebook variables to be used below
TARGET_CATALOG = os.environ.get("TARGET_CATALOG")
TARGET_SCHEMA = os.environ.get("TARGET_SCHEMA")
VS_INDEX = os.environ.get("VS_INDEX")

DATABRICKS_HOST = os.environ.get("DATABRICKS_HOST")
DATABRICKS_TOKEN = os.environ.get("DATABRICKS_TOKEN")
AI_GATEWAY_MODEL = os.environ.get("AI_GATEWAY_MODEL")
AI_GATEWAY_BASE_URL = os.environ.get("AI_GATEWAY_BASE_URL")
LLM_DESTINATION_MODEL = os.environ.get("LLM_DESTINATION_MODEL")
AGENT_LLM_SECRET_SCOPE = os.environ.get("AGENT_LLM_SECRET_SCOPE") or "weatherwise-agent"
MLFLOW_EXPERIMENT = os.environ.get("MLFLOW_EXPERIMENT")
RETRIEVER_TOOL_NAME = os.environ.get("RETRIEVER_TOOL_NAME")
AGENT_NAME = os.environ.get("AGENT_NAME")

MAILGUN_API_URL = os.environ.get("MAILGUN_API_URL")
MAILGUN_API_KEY = os.environ.get("MAILGUN_API_KEY")
SENDER = os.environ.get("SENDER")
RECIPIENT = os.environ.get("RECIPIENT")

SMS_ACCOUNT_SID = os.getenv("SMS_ACCOUNT_SID")
SMS_AUTH_TOKEN = os.getenv("SMS_AUTH_TOKEN")
SMS_ACCOUNT_ID = os.getenv("SMS_ACCOUNT_ID")  # AC... (needed when SMS_ACCOUNT_SID is an SK... API key)

print("Local variables:")
print(f" - TARGET_CATALOG: {TARGET_CATALOG}")
print(f" - TARGET_SCHEMA: {TARGET_SCHEMA}")
print(f" - DATABRICKS_HOST: {DATABRICKS_HOST}")
print(f" - DATABRICKS_TOKEN: {redact(DATABRICKS_TOKEN)}")
print(f" - AI_GATEWAY_MODEL: {AI_GATEWAY_MODEL}")
print(f" - AI_GATEWAY_BASE_URL: {AI_GATEWAY_BASE_URL}")
print(f" - LLM_DESTINATION_MODEL: {LLM_DESTINATION_MODEL}")
print(f" - AGENT_LLM_SECRET_SCOPE: {AGENT_LLM_SECRET_SCOPE}")
print(f" - MLFLOW_EXPERIMENT: {MLFLOW_EXPERIMENT}")
print(f" - VS_INDEX: {VS_INDEX}")
print(f" - RETRIEVER_TOOL_NAME: {RETRIEVER_TOOL_NAME}")
print(f" - AGENT_NAME: {AGENT_NAME}")
print(f" - MAILGUN_API_URL: {redact(MAILGUN_API_URL)}")
print(f" - MAILGUN_API_KEY: {redact(MAILGUN_API_KEY)}")
print(f" - SENDER: {SENDER}")
print(f" - RECIPIENT: {RECIPIENT}")
print(f" - SMS_ACCOUNT_SID: {redact(SMS_ACCOUNT_SID)}")
print(f" - SMS_AUTH_TOKEN: {redact(SMS_AUTH_TOKEN)}")
print(f" - SMS_ACCOUNT_ID: {redact(SMS_ACCOUNT_ID)}")

### 2.0 Create an MLflow Experiment

> Setting the experiment avoids MLflow using the default experiment name and keep things organized

In [0]:
import os
import mlflow
from mlflow.tracing.archival import enable_databricks_trace_archival

# Experiment name comes from MLFLOW_EXPERIMENT in .env, under your user folder.
user = spark.sql("select current_user()").first()[0]          # e.g. 'robert.leach@databricks.com'
exp_path = f"/Users/{user}/{MLFLOW_EXPERIMENT}"

mlflow.end_run()
experiment = mlflow.set_experiment(exp_path)
print("Experiment path:", exp_path, "| id:", experiment.experiment_id)

# Copy this experiment's traces (notebook runs + the served endpoint, which logs to the same
# experiment via MLFLOW_EXPERIMENT_ID) into ONE Unity Catalog Delta table. Archival runs
# periodically, so new traces appear in the table after a short delay.
TRACE_TABLE = os.environ.get("TRACE_TABLE") or f"{TARGET_CATALOG}.{TARGET_SCHEMA}.weatherwise_traces"
try:
    enable_databricks_trace_archival(delta_table_fullname=TRACE_TABLE, experiment_id=experiment.experiment_id)
    print("Trace archival ->", TRACE_TABLE)
except Exception as e:  # already enabled for this experiment, or archival unavailable
    print("Trace archival not changed:", str(e)[:300])


#### 2.1 Unit Test the Agent, tracing the chain of thought of the agent

In [ ]:
from weatherwise_agent import AGENT
from mlflow.types.responses import ResponsesAgentRequest

response = AGENT.predict(ResponsesAgentRequest(input=[
    {
        "role": "user",
        "content": "I heard the weather in New York is going to be cold tomorrow. Which in-transit shipments are at risk of temperature excursions, and what supplier escalation steps should I take? Also is there a backup supplier nearby? Email me a full report and send me an super concise SMS message with an abbreviated summary"
    }
]))

display(response)


### 2.2 Log the agent in MLflow (locally) before proceeding with evals

> Setting the experiment avoids MLflow using the default experiment name and keep things organized

> See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

In [ ]:
import os
import datetime
import mlflow
from weatherwise_agent import tools, AI_GATEWAY_MODEL
from databricks_langchain import VectorSearchRetrieverTool
from mlflow.models.resources import DatabricksFunction, DatabricksSQLWarehouse, DatabricksUCModelService
from unitycatalog.ai.langchain.toolkit import UnityCatalogTool

resources = []
for tool in tools:
    if isinstance(tool, VectorSearchRetrieverTool):
        resources.extend(tool.resources)
    elif isinstance(tool, UnityCatalogTool):
        resources.append(DatabricksFunction(function_name=tool.uc_function_name))
# UC functions run as SQL on this warehouse (fast; no serverless Spark session start-up).
if os.environ.get("WAREHOUSE_ID"):
    resources.append(DatabricksSQLWarehouse(warehouse_id=os.environ["WAREHOUSE_ID"]))
# Unity AI Gateway model service the agent's LLM calls go through (agent_src/setup_ai_gateway)
if AI_GATEWAY_MODEL and AI_GATEWAY_MODEL.count(".") == 2 and not AI_GATEWAY_MODEL.startswith("system."):
    resources.append(DatabricksUCModelService(model_service_name=AI_GATEWAY_MODEL))

input_example = {
    "input": [
        {
            "role": "user",
            "content": "I heard the weather in Philadelphia is going to be hot tomorrow. Which in-transit shipments are at risk of temperature excursions, and what supplier escalation steps should I take? Also is there a backup supplier nearby? Email me the results"
        }
    ]
}


with mlflow.start_run(run_name=f"{AGENT_NAME}-ut-{datetime.datetime.now():%Y%m%d}"):
    logged_agent_info = mlflow.pyfunc.log_model(
        name=AGENT_NAME,      # under model artifacts, the top level folder            
        python_model="weatherwise_agent.py",       # agent python file 
        input_example=input_example,
        resources=resources,
        code_paths=["tools/custom_tools/"],      
        # Declare runtime deps explicitly — MLflow's auto-inference misses these,
        # and the serving container needs them to load the agent.
        extra_pip_requirements=[
            "databricks-langchain>=0.19.0",
            "langchain-openai",
            "twilio",
            "python-dotenv",
        ]
    )

### 2.3 Load model into a wrapper function to make running evals easier (simplier syntax)

> Load the model and create a prediction function so you can ask the model a question without the boilerplate message: role: user: conetent: syntax

In [ ]:
# Load the model and create a prediction function to ask the model a question
# without the boilerplate of the Responses request/response schema.
import mlflow

# Resolve the logged model URI. Prefer the variable from the log_model cell (2.2);
# if that cell hasn't run in this session, fall back to the most recent logged-agent
# run in the experiment so this cell is safe to run on its own.
try:
    logged_model_uri = logged_agent_info.model_uri
except NameError:
    _user = spark.sql("select current_user()").first()[0]
    _exp = mlflow.set_experiment(f"/Users/{_user}/{MLFLOW_EXPERIMENT}")
    _runs = mlflow.search_runs(
        experiment_ids=[_exp.experiment_id],
        filter_string=f"attributes.run_name LIKE '{AGENT_NAME}-ut-%'",
        order_by=["start_time DESC"],
        max_results=1,
    )
    if len(_runs) == 0:
        raise RuntimeError(
            "No logged-agent run found. Run the 'Log the agent in MLflow' cell (2.2) first."
        )
    logged_model_uri = f"runs:/{_runs.iloc[0]['run_id']}/{AGENT_NAME}"

print("Using model URI:", logged_model_uri)
loaded_model = mlflow.pyfunc.load_model(logged_model_uri)

def predict_wrapper(query):
    # ResponsesAgent expects the OpenAI Responses input format
    model_input = {
        "input": [{"role": "user", "content": query}]
    }
    response = loaded_model.predict(model_input)

    # Pull the assistant text out of the Responses output items
    output = response["output"] if isinstance(response, dict) else response.output
    texts = []
    for item in output:
        item = item if isinstance(item, dict) else item.model_dump()
        if item.get("type") == "message":
            for part in item.get("content", []):
                if isinstance(part, dict) and part.get("type") in ("output_text", "text"):
                    texts.append(part.get("text", ""))
    return texts[-1] if texts else ""


### 2.4 Load eval data


In [ ]:
import pandas as pd

cases = [
    # Frozen Fibrin Sealant Kit (FIB-F04), Zimmer Biotech (ZB-SUP-2041)
    {
        "request": "According to Zimmer Biotech SOP ZB-SUP-2041, outline Tier-1, Tier-2, and Tier-3 escalation contacts and the SLA for the Frozen Fibrin Sealant Kit.",
        "expected_facts": [
            "Zimmer Biotech SOP ZB-SUP-2041 defines Tier 1-3 for the Frozen Fibrin Sealant Kit",
            "Tier 1: logistics@zimmerbio.com",
            "Tier 2: qa@zimmerbio.com; director.supply@zimmerbio.com",
            "Tier 3: vp.ops@zimmerbio.com | +1-312-555-0100",
            "SLA: Acknowledge 1h; quarantine decision 4h; CAPA 24h",
        ],
    },
    {
        "request": "Per ZB-SUP-2041, what storage range applies to the Frozen Fibrin Sealant Kit and what counts as an excursion?",
        "expected_facts": [
            "Keep frozen at -22 to -4°F (-30 to -20°C)",
            "Any reading above -4°F (-20°C) means the kit is thawing",
            "Do not refreeze; quarantine the lot until QA disposition",
            "No allowance: thawed kit is not released without QA sign-off",
        ],
    },

    # Cardiac Stent Kit (coated): MedPro Devices (MP-SLA-007) and MedAxis (escalation matrix)
    {
        "request": "Summarize MedPro Devices SLA MP-SLA-007 and the escalation path for the Cardiac Stent Kit (coated).",
        "expected_facts": [
            "MedPro MP-SLA-007 requires on-time ≥98%",
            "Storage 59–77°F (15–25°C); any reading above 77°F is a non-conformance",
            "Tier 2: operations@medpro.com (reading 78–86°F or delay >12h)",
            "Tier 3: ceo.office@medpro.com (reading above 86°F or STAT surgery at risk)",
            "Weather delay >12h: reroute through an alternate hub",
        ],
    },
    {
        "request": "Provide the MedAxis escalation matrix contacts (L1/L2/L3) for the Cardiac Stent Kit (coated).",
        "expected_facts": [
            "MedAxis Escalation Matrix defines L1/L2/L3 for the coated stent",
            "L1: shipping@medaxis.io (reading 78–86°F or delay <12h)",
            "L2: ops@medaxis.io (reading above 86°F or delay >12h)",
            "L3: exec@medaxis.io (STAT surgery at risk)",
            "Deviation report within 48h",
        ],
    },

    # Drug-Coated Balloon Catheter (paclitaxel): Ortho Dynamics (OD-QA-115)
    {
        "request": "For the Drug-Coated Balloon Catheter (paclitaxel) at Ortho Dynamics, list the L1/L2/L3 contacts, timing and the heat limit.",
        "expected_facts": [
            "Product: Drug-Coated Balloon Catheter (paclitaxel) (CATH-B08) | Supplier: Ortho Dynamics (OD-QA-115)",
            "Storage 59–86°F (15–30°C), ambient, out of direct sun",
            "Any reading above 86°F (30°C) is a heat excursion; isolate the lot",
            "L1: warehouse.alerts@orthodynamics.com (≤2h); L2: dist.ops@orthodynamics.com (≤4h); L3: cmo.office@orthodynamics.com (≤8h)",
        ],
    },

    # Refrigerated Thrombin Hemostat: CardioNova (CN-COLD-003)
    {
        "request": "List the CardioNova CN-COLD-003 tiers for the Refrigerated Thrombin Hemostat and what counts as freezing.",
        "expected_facts": [
            "Refrigerate at 36–46°F (2–8°C); do not freeze",
            "Readings 47–50°F need QA review",
            "Freezing (below 32°F) or above 50°F means rejection",
            "Tier 1: supply@cardionova.com; Tier 2: qa@cardionova.com; Tier 3: compliance@cardionova.com",
            "QA review within 24h",
        ],
    },

    # Bioprosthetic Aortic Valve (bovine tissue): ValveX Labs (VX-QMS-221)
    {
        "request": "According to ValveX Labs VX-QMS-221, how should a freeze event on the Bioprosthetic Aortic Valve be handled?",
        "expected_facts": [
            "Storage 50–77°F (10–25°C); never freeze",
            "Any reading below 32°F (0°C) is a freeze event; quarantine the valve at once",
            "No release without a ValveX written assessment",
            "Critical (below 32°F or above 86°F): escalate to qa.supervisor@valvex.com and compliance@valvex.com immediately",
            "SLA: acknowledge 2h; freeze quarantine within 1h",
        ],
    },
    {
        "request": "For the Bioprosthetic Aortic Valve (VALVE-E11) at ValveX Labs, how is a reading of 40°F classified and what is done?",
        "expected_facts": [
            "40°F falls in the Moderate band (32–44.9°F)",
            "Moderate: notify QA same shift and hold at destination",
            "QA disposition within 24h",
        ],
    },

    # Frozen Bone Allograft (cryopreserved): OrthoCore (Supplier Playbook)
    {
        "request": "Summarize the OrthoCore escalation tiers for the Frozen Bone Allograft (cryopreserved).",
        "expected_facts": [
            "Keep at or below -40°F (-40°C); allowed range -112 to -40°F",
            "Any reading above -40°F is a warming excursion; do not refreeze; quarantine the lot",
            "Tier 1: ops@orthocore.com (delay <24h, reading at or below -40°F)",
            "Tier 2: qa@orthocore.com (any reading above -40°F or delay >24h)",
            "Tier 3: reading above -22°F or STAT surgery at risk; request supplier executive review",
        ],
    },

    # Corneal Tissue Graft (cold-storage medium): SterileWorks (SW-QMS-118)
    {
        "request": "For the Corneal Tissue Graft (cold-storage medium) from SterileWorks, what are the L1/L2/L3 contacts and the freezing rule?",
        "expected_facts": [
            "Refrigerate at 36–46°F (2–8°C) in cold-storage medium; never freeze",
            "Freezing (below 32°F) or above 50°F means the graft is rejected",
            "L1: h.price@sterileworks.com (≤2h)",
            "L2: tissue.qa@sterileworks.com (≤4h)",
            "L3: ops.director@sterileworks.com | +1-415-555-0162 (≤8h)",
        ],
    },

    # Cross-product storage: the ranges differ by product, so no single 2-8°C rule
    {
        "request": "What storage temperature applies to each product in these SOPs?",
        "expected_facts": [
            "Frozen products: fibrin sealant -22 to -4°F; bone allograft -112 to -40°F",
            "Refrigerated 36–46°F (2–8°C): thrombin hemostat and corneal graft",
            "Tissue valve 50–77°F (10–25°C), must never freeze; coated stent 59–77°F; drug-coated balloon 59–86°F",
        ],
    },
]

eval_df = pd.DataFrame(cases)

# Format for mlflow.genai.evaluate with expected_facts as a list for Correctness scorer
eval_data = [
    {
        "inputs": {"query": row["request"]},
        "expectations": {
            "expected_facts": row["expected_facts"]
        },
    }
    for _, row in eval_df.iterrows()
]

### 2.5 Define LLM Judges


In [0]:
from mlflow.genai.scorers import (
    Correctness,         # built-in judge: checks factual accuracy vs expectations.expected_facts
    Safety,              # built-in safety scorer
    RelevanceToQuery,    # built-in relevance scorer
    Guidelines           # LLM-judge wrapper for your custom tone/style rules
)

scorers = [
    Correctness(),        # factual accuracy judge
    Safety(),             # checks harmful/off-policy content
    RelevanceToQuery(),   # measures response relevance to query
    Guidelines(           # your custom style/tone scorer
        name="jj_tone",
        guidelines="""Professional and concise:
        • No chit-chat or clarifying questions
        • No logs/JSON/system/meta
        Pass if clear, factual, and structured; fail if casual/wordy or asks questions."""
    ),
]

### 2.6 Run evals


In [0]:
print("Running evaluation...")
with mlflow.start_run(run_name=f"{AGENT_NAME}-evals-{datetime.datetime.now():%Y%m%d}"):
    results = mlflow.genai.evaluate(
        data=eval_data,
        predict_fn=predict_wrapper, 
        scorers=scorers,
    )

#### 2.6 Interate
>Based on evals go back to the agent code (.py file) and change our prompt to be more specific, more guildines, reduce marketing fluff, be more actionable, etc.

### 2.7 After satisified with evals, log model to Unity Catalog


In [0]:
from databricks.sdk import WorkspaceClient
import os

uc_model_name = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{AGENT_NAME}"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=uc_model_name)

In [0]:
from IPython.display import display, HTML

# Retrieve the Databricks host URL
workspace_url = spark.conf.get('spark.databricks.workspaceUrl')

# Create HTML link to created agent
html_link = f'<a href="https://{workspace_url}/explore/data/models/{TARGET_CATALOG}/{TARGET_SCHEMA}/{AGENT_NAME}" target="_blank">Go to Unity Catalog to see Registered Agent</a>'
display(HTML(html_link))

### 2.8 Deploy to model serving (and spin up human feedback app)


In [0]:
print("Local model URI:", logged_agent_info.model_uri)
print("Local model Run ID:", logged_agent_info.run_id)
print("UC Registered Model Version:", uc_registered_model_info.version)

In [0]:
from databricks import agents

# Deploy the model to the review app and a model serving endpoint


from databricks.sdk import WorkspaceClient
workspace_host = WorkspaceClient().config.host

# No PAT on the endpoint: tools use Model Serving automatic auth passthrough (declared
# resources); the LLM calls run as the dedicated service principal created by
# scripts/setup_agent_llm_sp.sh. {{secrets/...}} references resolve at serving time.
dbutils.secrets.get(AGENT_LLM_SECRET_SCOPE, "sp-client-secret")  # fail fast if the scope isn't set up
environment_vars = {
    # Send endpoint traces to THIS experiment. Without it, agents.deploy keeps the ID the
    # endpoint was first created with (a since-deleted experiment).
    "MLFLOW_EXPERIMENT_ID": experiment.experiment_id,
    "TARGET_CATALOG": TARGET_CATALOG,
    "TARGET_SCHEMA": TARGET_SCHEMA,
    "VS_INDEX": VS_INDEX,
    "AI_GATEWAY_MODEL": AI_GATEWAY_MODEL,
    "AI_GATEWAY_BASE_URL": AI_GATEWAY_BASE_URL,
    "LLM_DESTINATION_MODEL": LLM_DESTINATION_MODEL,
    "LLM_HOST": workspace_host,
    "LLM_CLIENT_ID": "{{secrets/" + AGENT_LLM_SECRET_SCOPE + "/sp-client-id}}",
    "LLM_CLIENT_SECRET": "{{secrets/" + AGENT_LLM_SECRET_SCOPE + "/sp-client-secret}}",
    "RETRIEVER_TOOL_NAME": RETRIEVER_TOOL_NAME,
    "WAREHOUSE_ID": os.environ.get("WAREHOUSE_ID"),  # UC functions execute on this SQL warehouse
    "AGENT_NAME": AGENT_NAME,  # From: name on agent emails (tool_send_email)
    "MAILGUN_API_URL": MAILGUN_API_URL,
    "MAILGUN_API_KEY": MAILGUN_API_KEY,
    "SENDER": SENDER,
    "RECIPIENT": RECIPIENT,
    "SMS_ACCOUNT_SID": SMS_ACCOUNT_SID,
    "SMS_AUTH_TOKEN": SMS_AUTH_TOKEN,
    "SMS_ACCOUNT_ID": SMS_ACCOUNT_ID,
    "LINKEDIN_URL": os.environ.get("LINKEDIN_URL"),  # send_connect_sms profile link (optional override)
}
# Model Serving rejects null environment variable values.
environment_vars = {k: v for k, v in environment_vars.items() if v}

agents.deploy(
    model_name=uc_model_name,
    model_version=uc_registered_model_info.version,
    # Demo days: keep a container warm so the first question isn't a cold start.
    # Set AGENT_SCALE_TO_ZERO=true in .env to save cost when not demoing.
    scale_to_zero=os.environ.get("AGENT_SCALE_TO_ZERO", "false").lower() == "true",
    environment_vars=environment_vars,
    tags={"endpointSource": AGENT_NAME},
)